# 9. Transformer
**Use case:** Insurance — Claim notes / first notice of loss text triage

When a policyholder reports a loss, the call-centre or app captures a short free-text note such as 'windshield cracked by rock on highway' or 'car rolled, airbags deployed, driver taken to hospital'. Today a triage desk reads every note and routes it by hand to Auto-Glass, Total-Loss, Injury or Fraud-review queues, which delays simple glass claims and lets injury claims wait. The insurer has thousands of historic notes with the queue each one finally went to. Success means routing most claims automatically within seconds at high accuracy, sending low-confidence and fraud-flagged claims to people, and showing which words drove each decision.

> 💡 Picture a claims adjuster reading a note with a highlighter in each hand. For every word, they glance at all the other words and highlight the ones that change its meaning: 'cracked' looks at 'windshield', 'not' looks at 'injured'. Self-attention is that highlighting, done by many readers (heads) in parallel.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Compute self-attention by hand on a 4-word claim note
Calculate scaled dot-product self-attention in numpy for 'windshield cracked by rock' and read the attention weights.

| 5W + H | |
|---|---|
| **Who** | A claims operations lead or architect who wants to understand what 'attention' means. |
| **What** | Four words with hand-made 4-number embeddings and small query, key and value matrices. |
| **When** | Before evaluating vendor or in-house Transformer models for FNOL triage. |
| **Where** | A numpy-only script; no deep-learning library needed. |
| **Why** | Attention is the core idea behind every Transformer and LLM; seeing the numbers removes the mystery. |
| **How** | Build Q, K, V with matrix multiplication, compute Q times K transpose divided by square root of d, apply softmax per row and multiply by V. |

**🧠 ML concepts:** Embeddings as vectors, Query, key and value, Scaled dot-product attention, Softmax as normalized weights, Context-aware token vectors  
**🐍 Python concepts:** Dictionaries mapping words to lists, List comprehension to build a matrix, The @ matrix-multiply operator, Defining a function with keepdims broadcasting, zip to print labelled rows, str.join with a generator

In [ ]:
import numpy as np

np.random.seed(42)
# 🐍 PY: a short claim note split into a list of words (tokens)
tokens = ["windshield", "cracked", "by", "rock"]
# 🧠 ML: each word becomes a small vector (embedding); real models learn 512+ numbers per word
# 🐍 PY: a dict maps each word to a hand-made 4-number vector
emb = {
    "windshield": [1.0, 0.9, 0.0, 0.1],
    "cracked":    [0.8, 1.0, 0.1, 0.0],
    "by":         [0.1, 0.0, 0.2, 0.1],
    "rock":       [0.6, 0.5, 0.0, 0.3],
}
# 🐍 PY: list comprehension pulls each token's vector in order -> 4x4 matrix
X = np.array([emb[t] for t in tokens])
d = X.shape[1]

# 🧠 ML: Q (query) = what am I looking for, K (key) = what do I offer, V (value) = what I pass on
# 🧠 ML: in a trained model W_q, W_k, W_v are learned; here they are small random matrices
W_q = np.eye(d) + 0.1 * np.random.randn(d, d)
W_k = np.eye(d) + 0.1 * np.random.randn(d, d)
W_v = np.eye(d)
# 🐍 PY: the @ operator is matrix multiplication
Q, K, V = X @ W_q, X @ W_k, X @ W_v

# 🧠 ML: score = how well each word's query matches every other word's key; divide by sqrt(d) to keep calm
scores = Q @ K.T / np.sqrt(d)
print("Raw attention scores (row = word looking, col = word looked at):")
# 🐍 PY: np.round + f-string padding prints a neat table
for t, row in zip(tokens, np.round(scores, 2)):
    print(f"  {t:>10s} {row}")

# 🧠 ML: softmax turns each row into percentages that sum to 1 (the attention weights)
def softmax(z):
    # 🐍 PY: keepdims=True keeps the shape (4,1) so broadcasting divides row by row
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

A = softmax(scores)
print("\nAttention weights (each row sums to 1):")
for t, row in zip(tokens, A):
    # 🐍 PY: nested f-string formatting inside a join builds one aligned line
    print(f"  {t:>10s} " + " ".join(f"{w:.2f}" for w in row))

# 🧠 ML: output = weighted blend of value vectors -> each word now carries context from the others
out = A @ V
print("\nContext-aware vector for 'cracked':", np.round(out[1], 2))
# 🧠 ML: crude routing - compare the sentence average to a 'glass' direction vector
glass_dir = np.array([1, 1, 0, 0]) / np.sqrt(2)
print(f"Similarity to Auto-Glass queue: {out.mean(axis=0) @ glass_dir:.2f}")
print("Word 'by' receives the least attention:", tokens[int(A.sum(axis=0).argmin())])

**Try this next:**
- Change the embedding of 'rock' to [0, 0, 1, 0] and see how attention to it changes.
- Remove the division by np.sqrt(d) and observe the weights become sharper.
- Add a fifth word 'injured' with vector [0, 0, 1, 1] and check the glass similarity.

## BUILD: Train a tiny Transformer encoder to route 2,000 FNOL notes
Build and train a one-layer PyTorch Transformer encoder that classifies synthetic claim notes into four queues.

| 5W + H | |
|---|---|
| **Who** | A data scientist on the claims analytics team. |
| **What** | 2,000 synthetic 8-word FNOL notes with queue labels, 5 percent of labels flipped to mimic mis-routing. |
| **When** | During a proof of concept before investing in a larger pre-trained model. |
| **Where** | A CPU-only Python environment with PyTorch. |
| **Why** | To learn the moving parts (vocab, embedding, encoder layer, pooling, head, training loop) on a problem small enough to run in seconds. |
| **How** | Build a word vocab, encode notes as id tensors, define an nn.Module with Embedding plus TransformerEncoderLayer plus Linear, train with Adam for 6 epochs and score a new note. |

**🧠 ML concepts:** Tokenization and vocabulary, Embedding layer, Transformer encoder layer (attention plus feed-forward), Mean pooling for classification, Mini-batch training with cross-entropy, Label noise limits accuracy  
**🐍 Python concepts:** Classes inheriting from nn.Module, super().__init__(), Set and dict comprehensions, random.sample, choices and shuffle, with torch.no_grad() context manager, dict.get with a default

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
random.seed(42)
QUEUES = ["AutoGlass", "TotalLoss", "Injury", "FraudReview"]

# 🐍 PY: dict of lists - each queue has its own typical words; 'common' words appear everywhere
words = {
    "AutoGlass": ["windshield", "chip", "cracked", "glass", "rock", "window", "sunroof"],
    "TotalLoss": ["flooded", "fire", "rolled", "frame", "airbags", "towed", "destroyed"],
    "Injury": ["neck", "ambulance", "hospital", "pain", "whiplash", "er", "back"],
    "FraudReview": ["cash", "witness", "lapsed", "late", "receipts", "inconsistent", "prior"],
}
common = ["vehicle", "insured", "reported", "on", "highway", "at", "night", "the", "car", "damage"]


def make_note(q, length=8):
    # 🧠 ML: 2 queue words + 1-2 words from other queues (noise) + filler -> realistic messiness
    rivals = [x for x in QUEUES if x != q]
    others = [random.choice(words[random.choice(rivals)]) for _ in range(random.randint(1, 2))]
    toks = random.sample(words[q], 2) + others
    toks += random.choices(common, k=length - len(toks))
    random.shuffle(toks)
    return toks


# 🐍 PY: list comprehension of (tokens, label_index) tuples - 2000 synthetic FNOL notes
data = [(make_note(q), i) for _ in range(500) for i, q in enumerate(QUEUES)]
# 🧠 ML: 5% of labels flipped at random = adjusters sometimes mis-route; caps achievable accuracy
data = [(t, random.randrange(4) if random.random() < 0.05 else lab) for t, lab in data]
random.shuffle(data)
# 🐍 PY: set union of all words, then enumerate to give each a number; 0 is reserved for padding
vocab = {w: i + 1 for i, w in enumerate(sorted({w for toks, _ in data for w in toks}))}
X = torch.tensor([[vocab[w] for w in toks] for toks, _ in data])
y = torch.tensor([lab for _, lab in data])
X_tr, y_tr, X_te, y_te = X[:1600], y[:1600], X[1600:], y[1600:]
print(f"Vocab size={len(vocab)}  train={len(X_tr)}  test={len(X_te)}  example={data[0][0]}")


# 🐍 PY: a class that inherits nn.Module; __init__ builds layers, forward() runs them
class TinyClaimTransformer(nn.Module):
    def __init__(self, vocab_size, d=32, heads=2, n_cls=4):
        super().__init__()
        # 🧠 ML: embedding = lookup table turning word ids into learnable vectors
        self.emb = nn.Embedding(vocab_size + 1, d, padding_idx=0)
        # 🧠 ML: one encoder layer = multi-head self-attention + small feed-forward net
        self.enc = nn.TransformerEncoderLayer(d, heads, dim_feedforward=64, batch_first=True, dropout=0.1)
        self.head = nn.Linear(d, n_cls)

    def forward(self, x):
        h = self.enc(self.emb(x))
        # 🧠 ML: mean-pool all word vectors into one note vector, then score each queue
        return self.head(h.mean(dim=1))


model = TinyClaimTransformer(len(vocab))
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(6):
    model.train()
    # 🐍 PY: torch.randperm shuffles indices; range(start, stop, step) walks mini-batches of 64
    perm = torch.randperm(len(X_tr))
    for i in range(0, len(X_tr), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = loss_fn(model(X_tr[idx]), y_tr[idx])
        loss.backward()
        opt.step()
    model.eval()
    # 🐍 PY: no_grad() turns off gradient tracking for faster, safer evaluation
    with torch.no_grad():
        acc = (model(X_te).argmax(1) == y_te).float().mean().item()
    print(f"epoch {epoch + 1}  last-batch loss={loss.item():.3f}  test acc={acc:.3f}")

with torch.no_grad():
    pred = model(X_te).argmax(1)
print("\nPer-queue test accuracy:")
for i, q in enumerate(QUEUES):
    m = y_te == i
    print(f"  {q:12s} {(pred[m] == i).float().mean().item():.2f}  (n={int(m.sum())})")

new = "insured reported neck pain after car rolled on highway".split()
# 🐍 PY: dict.get(w, 0) returns 0 for unknown words instead of raising KeyError
ids = torch.tensor([[vocab.get(w, 0) for w in new]])
with torch.no_grad():
    probs = torch.softmax(model(ids), dim=1)[0]
print("\nNew FNOL note:", " ".join(new))
for q, p in sorted(zip(QUEUES, probs.tolist()), key=lambda t: -t[1]):
    print(f"  {q:12s} {p:.2f} {'#' * int(p * 30)}")

**Try this next:**
- Change heads=2 to heads=4 and d=32 to d=16 and compare accuracy and speed.
- Raise label noise from 0.05 to 0.20 and watch test accuracy fall.
- Write your own note such as 'rock chipped windshield, cash only repair shop' and see how the model splits Glass vs Fraud.

## PRODUCTION: FNOL routing pipeline with masking, explanations, baseline and governance
Build a fuller Transformer claim router with tokenizer, positional encoding, padding mask, 2-layer multi-head encoder, attention-pooling explanations, early stopping, a TF-IDF baseline and a human-in-the-loop threshold.

| 5W + H | |
|---|---|
| **Who** | The claims AI team, the claims operations owner and model governance. |
| **What** | 2,400 variable-length synthetic notes (6-16 words) with a 'not injured' phrase that depends on word order, and 5 percent label noise. |
| **When** | Pre-production, deciding whether the Transformer is worth deploying over a simpler model. |
| **Where** | A CPU training job; the saved model and vocab are served behind the claims system's routing API. |
| **Why** | Fast correct routing cuts glass cycle time and gets injured claimants help sooner, while regulators expect explainable, monitored AI decisions. |
| **How** | Tokenizer class with pad and unknown ids, sine-cosine positional encoding, nn.TransformerEncoder with src_key_padding_mask, attention pooling whose weights explain decisions, train/val/test with best-epoch selection, TF-IDF plus logistic regression baseline, confidence threshold and torch.save. |

**🧠 ML concepts:** Tokenizer with padding and unknown tokens, Positional encoding, Multi-head self-attention via nn.TransformerEncoder, Padding masks, Attention pooling as a (partial) explanation, Baseline comparison and human-in-the-loop thresholds  
**🐍 Python concepts:** Classes with __init__ and methods, register_buffer for non-trained tensors, Slice assignment to insert into a list, map with tuple unpacking, lambda functions, Dict comprehension to copy model state

In [ ]:
import os
import math
import random
import tempfile
import warnings
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")
torch.set_num_threads(1)
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
QUEUES = ["AutoGlass", "TotalLoss", "Injury", "FraudReview"]
PAD, UNK, MAXLEN = "<pad>", "<unk>", 16

KEYWORDS = {
    "AutoGlass": ["windshield", "chip", "cracked", "glass", "rock", "window", "sunroof", "wiper"],
    "TotalLoss": ["flooded", "fire", "rolled", "frame", "airbags", "towed", "destroyed", "submerged"],
    "Injury": ["neck", "ambulance", "hospital", "pain", "whiplash", "er", "back", "concussion"],
    "FraudReview": ["cash", "lapsed", "receipts", "inconsistent", "prior", "unwitnessed", "midnight", "new"],
}
FILLER = "vehicle insured reported on highway at the car damage driver called claimant after parked".split()

def make_note(q):
    # 🧠 ML: variable-length notes (6-16 words) force us to pad and mask - like real FNOL text
    n = random.randint(6, MAXLEN)
    rivals = [x for x in QUEUES if x != q]
    toks = random.sample(KEYWORDS[q], 2) + [random.choice(KEYWORDS[random.choice(rivals)])]
    toks += random.choices(FILLER, k=max(0, n - len(toks) - 2))
    random.shuffle(toks)
    # 🧠 ML: word ORDER matters for one phrase - 'not injured' must NOT be routed to Injury
    phrase = ["driver", "injured"] if q == "Injury" else (["not", "injured"] if random.random() < 0.4 else [])
    # 🐍 PY: slice assignment toks[i:i] = [...] inserts a list at position i without replacing
    i = random.randint(0, len(toks))
    toks[i:i] = phrase
    return toks[:MAXLEN]

# 🐍 PY: a nested comprehension - outer loop repeats n_per times, inner loop walks every queue
def build_data(n_per=600):
    data = [(make_note(q), i) for _ in range(n_per) for i, q in enumerate(QUEUES)]
    # 🧠 ML: 5% label noise = real-world mis-routing by adjusters
    data = [(t, random.randrange(4) if random.random() < 0.05 else y) for t, y in data]
    random.shuffle(data)
    return data

class Tokenizer:
    """Word-level vocab. Real systems use sub-word BPE/WordPiece like GPT/BERT."""
    def __init__(self, notes):
        words = sorted({w for toks in notes for w in toks})
        # 🧠 ML: id 0 = padding, id 1 = unknown word (handles words never seen in training)
        self.itos = [PAD, UNK] + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, toks):
        # 🐍 PY: [0] * k makes a list of k zeros - the padding to a fixed length
        ids = [self.stoi.get(w, 1) for w in toks][:MAXLEN]
        return ids + [0] * (MAXLEN - len(ids))

class PositionalEncoding(nn.Module):
    # 🧠 ML: attention alone ignores order; sine/cosine waves stamp each slot with its position
    def __init__(self, d, maxlen=MAXLEN):
        super().__init__()
        pos = torch.arange(maxlen).unsqueeze(1)
        # 🧠 ML: each pair of dimensions uses a different wave length, from fast to very slow
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe = torch.zeros(maxlen, d)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]

class ClaimRouter(nn.Module):
    def __init__(self, vocab, d=32, heads=4, layers=2, n_cls=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=0)
        self.pos = PositionalEncoding(d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=64, dropout=0.1, batch_first=True)
        # 🧠 ML: stack of encoder layers; each = multi-head self-attention + feed-forward + residual
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        # 🧠 ML: attention pooling - one learned score per word; weights double as an explanation
        self.pool = nn.Linear(d, 1)
        self.head = nn.Linear(d, n_cls)

    def forward(self, x, return_attn=False):
        # 🐍 PY: comparing a tensor to 0 gives a True/False tensor of the same shape
        mask = x == 0
        # 🧠 ML: src_key_padding_mask tells attention to ignore the <pad> slots
        h = self.encoder(self.pos(self.emb(x)), src_key_padding_mask=mask)
        # 🧠 ML: masked_fill sets padding scores to -1e9 so softmax gives them ~0 weight
        s = self.pool(h).squeeze(-1).masked_fill(mask, -1e9)
        w = torch.softmax(s, dim=1)
        logits = self.head((w.unsqueeze(-1) * h).sum(1))
        return (logits, w) if return_attn else logits

def train(model, Xtr, ytr, Xva, yva, epochs=10):
    # 🧠 ML: AdamW = Adam with weight decay, a standard regulariser for Transformers
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
    loss_fn = nn.CrossEntropyLoss()
    best_f1, best_state = -1, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            # 🧠 ML: backward() computes gradients; step() nudges every weight to reduce the loss
            loss_fn(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            f1 = f1_score(yva, model(Xva).argmax(1), average="macro")
        # 🧠 ML: keep the best epoch on validation = simple early stopping against overfitting
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  epoch {ep + 1:2d}  val macro-F1={f1:.3f}")
    model.load_state_dict(best_state)
    return best_f1

data = build_data()
n = len(data)
tr, va, te = data[: int(.7 * n)], data[int(.7 * n): int(.85 * n)], data[int(.85 * n):]
# 🧠 ML: build the vocab from TRAINING notes only so test words cannot leak in
tok = Tokenizer([t for t, _ in tr])

def to_tensor(rows):
    return torch.tensor([tok.encode(t) for t, _ in rows]), torch.tensor([y for _, y in rows])

# 🐍 PY: map applies to_tensor to each split; tuple unpacking names all six results at once
(Xtr, ytr), (Xva, yva), (Xte, yte) = map(to_tensor, (tr, va, te))
print(f"notes train/val/test = {len(tr)}/{len(va)}/{len(te)}  vocab={len(tok.itos)}")

model = ClaimRouter(len(tok.itos))
print(f"params={sum(p.numel() for p in model.parameters()):,}")
train(model, Xtr, ytr, Xva, yva)
with torch.no_grad():
    pred = model(Xte).argmax(1)
tf_f1 = f1_score(yte, pred, average="macro")

# 🧠 ML: baseline - TF-IDF bag of words + logistic regression; bigrams give it some word order
vec = TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"\S+")
join = lambda rows: [" ".join(t) for t, _ in rows]
lr = LogisticRegression(max_iter=2000, C=5).fit(vec.fit_transform(join(tr)), ytr.numpy())
lr_f1 = f1_score(yte, lr.predict(vec.transform(join(te))), average="macro")
print(f"\nTest macro-F1  Transformer={tf_f1:.3f}   TF-IDF+LogReg baseline={lr_f1:.3f}")
print("Lesson: on short keyword notes a cheap baseline can match or beat it;"
      " Transformers pay off on long, messy text.")
print(classification_report(yte, pred, target_names=QUEUES, digits=2))

# 🧠 ML: explanation - which words got the most pooling attention for one routed claim
note = "claimant reported windshield cracked by rock on highway driver not injured".split()
with torch.no_grad():
    logits, w = model(torch.tensor([tok.encode(note)]), return_attn=True)
probs = torch.softmax(logits, 1)[0]
print("Note:", " ".join(note))
print("Route ->", QUEUES[int(probs.argmax())], f"(p={probs.max():.2f})  top words by attention:")
for word, a in sorted(zip(note, w[0][: len(note)].tolist()), key=lambda t: -t[1])[:4]:
    print(f"  {word:12s} {a:.2f} {'#' * int(a * 40)}")

# 🧠 ML: business rule - low confidence or FraudReview goes to a human adjuster (human-in-the-loop)
with torch.no_grad():
    conf = torch.softmax(model(Xte), 1).max(1).values
# 🧠 ML: the 0.85 threshold trades automation rate against accuracy; tune with claims ops
auto = conf >= 0.85
print(f"\nAuto-route {auto.float().mean():.0%} of claims at accuracy "
      f"{(pred[auto] == yte[auto]).float().mean():.3f}; rest go to triage desk.")
# Governance: LLMs (GPT-style decoder Transformers) can do this zero-shot, but insurers must
# document data lineage, bias tests and explainability under NAIC AI Model Bulletin + model-risk policy.
print("Governance: log model version, attention explanations and overrides for NAIC AI audit.")
# 🧠 ML: save vocab and queue names WITH the weights - the model is useless without them
path = os.path.join(tempfile.mkdtemp(), "claim_router.pt")
torch.save({"state": model.state_dict(), "vocab": tok.itos, "queues": QUEUES}, path)
print("Saved ->", os.path.basename(path))

**Try this next:**
- Remove the positional encoding (return x in forward) and compare test F1 on notes containing 'not injured'.
- Make notes longer (MAXLEN=40) with more filler and cross-queue words and see whether the Transformer overtakes TF-IDF.
- Change the 0.85 threshold to 0.95 and report the auto-route share vs accuracy trade-off.